# Python System Interpreter

This notebook introduces a common Python packaging problem on a shared Linux machine. Bob and Alice both work on the same computer and both rely on the same system interpreter. Bob is the administrator for this shared machine, so he can install or replace packages in the system Python environment. Alice is a normal user, so she cannot modify system packages directly. This difference matters because the Python interpreter is shared, but the users and their home directories are separate.

| User    | Home           | Sudo |
| ------- | -------------- | ---- |
| `bob`   | `/home/bob`    | yes  |
| `alice` | `/home/alice`  | no   |

This arrangement creates dependency conflicts quickly. A package change made for one user can break the other user's project, and Alice's own work becomes hard to manage when her multiple projects need different versions of the same dependency. One shared Python environment cannot safely satisfy all of those conflicting requirements at the same time.

This notebook focuses on these **dependency-conflict scenarios**:

- **Shared machine conflict.** Two users share one system Python installation.
- **Cross-project conflict.** Alice's own projects can still fight over incompatible versions when they share one user-level environment.
- **Isolation progression.** We then compare system packages, `pip install --user`, and project-specific virtual environments.

---

## Inspect the Environment

### Inspect Registered Users

First, verify that `alice` and `bob` have been registered as specified in the `Dockerfile`.

In [2]:
%%bash
echo "-- User Information --"
echo "Bob:"
getent passwd bob | awk -F: '{print "user="$1, "uid="$3}'
echo ""
echo "Alice:"
getent passwd alice | awk -F: '{print "user="$1, "uid="$3}'

-- User Information --
Bob:
user=bob uid=1000

Alice:
user=alice uid=1001


Then, verify that `bob`, as specified in the `devcontainer.json`, is the default user. Additionally, we check for his elevated `sudo` rights.

In [5]:
%%bash
echo "This Jupyter Notebook runs as '$(whoami)'"

This Jupyter Notebook runs as 'bob'


### Explore System Interpreter

First, identify the shared *System Interpreter* itself by confirming the version that both `bob` and `alice` inherit before any user-level or project-level isolation is introduced.

In [9]:
%%bash
echo "This Jupyter Notebook uses '$(which python3)' and '$(python3 --version)' as Python version"

This Jupyter Notebook uses '/usr/bin/python3' and 'Python 3.10.12' as Python version


Next, verify where the Python interpreter looks for installed packages for Bob by showing the interpreter’s import search path (`sys.path`), the user-specific package locations, and whether the user site is enabled.

In [10]:
%%bash
python3 -m site

sys.path = [
    '/workspace',
    '/usr/lib/python310.zip',
    '/usr/lib/python3.10',
    '/usr/lib/python3.10/lib-dynload',
    '/usr/local/lib/python3.10/dist-packages',
    '/usr/lib/python3/dist-packages',
]
USER_BASE: '/home/bob/.local' (doesn't exist)
USER_SITE: '/home/bob/.local/lib/python3.10/site-packages' (doesn't exist)
ENABLE_USER_SITE: True


Then, we perform the same check for `alice` by running the `python3 -m site` command in her user context.

In [11]:
%%bash
sudo su - alice -c "python3 -m site"

sys.path = [
    '/home/alice',
    '/usr/lib/python310.zip',
    '/usr/lib/python3.10',
    '/usr/lib/python3.10/lib-dynload',
    '/home/alice/.local/lib/python3.10/site-packages',
    '/usr/local/lib/python3.10/dist-packages',
    '/usr/lib/python3/dist-packages',
]
USER_BASE: '/home/alice/.local' (exists)
USER_SITE: '/home/alice/.local/lib/python3.10/site-packages' (exists)
ENABLE_USER_SITE: True


### System Target Packages

Linux supports installing Python packages through the operating system's package manager, such as APT on Ubuntu. In this example, line 31 of the `Dockerfile` installs `python3-systemd` using `apt`. This is a suitable approach for packages that are closely integrated with the operating system because APT can manage both the Python package and its system-level dependencies.

In [12]:
%%bash
apt list --installed | grep python3-systemd

python3-systemd/now 234-3ubuntu2 amd64 [installed,local]


The package is now available to Python and appears in the list of installed packages. We can verify this with `pip list`, which displays the Python packages installed in the current environment. Filtering the output with `grep` makes it easy to find the `systemd-python` package.

In [13]:
%%bash
python3 -m pip list | grep systemd-python

systemd-python            234


Those OS-controlled Python packages are typically installed into the system package directory. On Ubuntu, packages managed by APT are placed under `/usr/lib/python3/dist-packages/`.

In [14]:
%%bash
ls -lah /usr/lib/python3/dist-packages/

total 48K
drwxr-xr-x 11 root root 4.0K Sep  1 10:47 .
drwxr-xr-x  3 root root 4.0K Sep  1 10:46 ..
drwxr-xr-x  3 root root 4.0K Sep  1 10:47 _distutils_hack
drwxr-xr-x  5 root root 4.0K Sep  1 10:47 pip
drwxr-xr-x  2 root root 4.0K Sep  1 10:47 pip-22.0.2.dist-info
drwxr-xr-x  6 root root 4.0K Sep  1 10:47 pkg_resources
drwxr-xr-x  7 root root 4.0K Sep  1 10:47 setuptools
drwxr-xr-x  2 root root 4.0K Sep  1 10:47 setuptools-59.6.0.egg-info
drwxr-xr-x  4 root root 4.0K Sep  1 10:47 systemd
-rw-r--r--  1 root root  586 Mar 17  2022 systemd_python-234.egg-info
drwxr-xr-x  5 root root 4.0K Sep  1 10:47 wheel
drwxr-xr-x  2 root root 4.0K Sep  1 10:47 wheel-0.37.1.egg-info


### Local Admin Target Packages

Packages installed by local administrators are typically placed under `/usr/local` and are visible to every user of the system interpreter. The notebook tooling may bring Requests in as a transitive dependency, so the command below reports the system copy without relying on a particular version.

In [15]:
%%bash
python3 -m pip list | grep requests

requests                  2.25.1


The user `alice` has also access to the dedicated package.

In [16]:
%%bash
sudo su - alice -c 'python3 -m pip list | grep requests'

requests                  2.32.3


On Ubuntu, the local admin Python package directory is typically located under `/usr/local/lib/pythonX.Y/dist-packages/` and, again, visible for all local users. The exact `X.Y` segment depends on the Python version shipped in the container.

In [17]:
%%bash
LOCAL_ADMIN_SITE=$(python3 -c "import sys; print(f'/usr/local/lib/python{sys.version_info.major}.{sys.version_info.minor}/dist-packages')")
echo "$LOCAL_ADMIN_SITE"
ls -lah "$LOCAL_ADMIN_SITE" | grep requests

/usr/local/lib/python3.10/dist-packages
drwxr-xr-x  3 root root 4.0K Sep  1 11:22 requests
drwxr-xr-x  2 root root 4.0K Sep  1 11:22 requests-2.25.1.dist-info


### User Target Packages

In [20]:
%%bash
echo "The User '$(whoami)' has the following Python packages installed in their user site:"
python3 -m pip list --user

The User 'bob' has the following Python packages installed in their user site:


In [21]:
%%bash
echo "The User 'alice' has the following Python packages installed in their user site:"
sudo su - alice -c 'python3 -m pip list --user | grep fastapi'

The User 'alice' has the following Python packages installed in their user site:
fastapi            0.68.2


---

## Setting Up the Demo Projects

This shared machine hosts **Bob's Legacy Application**, **Alice FastAPI v1**, and **Alice FastAPI v2**. The projects expose two conflicts:

1. Bob's application requires Requests 2.25.1, while Alice's projects require Requests 2.27 or newer for `JSONDecodeError`.
2. Alice FastAPI v1 uses Pydantic v1, while Alice FastAPI v2 uses Pydantic v2's `model_dump()` method.

### Bob's Legacy Application

Bob's legacy application is a small command-line program that requires Requests 2.25.1. It accepts a `--domain` argument and prepares a GET request for that address.

The following `%%bash` cell creates the application directly in Bob's project directory.

In [22]:
%%bash

BOB_LEGACY="/home/bob/projects/legacy-project"
BOB_MAIN="${BOB_LEGACY}/main.py"

mkdir -p "${BOB_LEGACY}"

cat > "${BOB_MAIN}" <<'PYTHON'
# Bob's legacy application.
import argparse
import sys

import requests


def create_request(domain):
    """Prepare the request used by the legacy application."""
    return requests.Request(method="GET", url=domain)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--domain", default="https://example.com")
    args = parser.parse_args()

    print("Bob Legacy Application")
    print(f"  Python  : {sys.executable}")
    print(f"  requests: {requests.__version__}")

    request = create_request(args.domain)
    print(f"  request : {request.method} {request.url}")


if __name__ == "__main__":
    main()
PYTHON

chown -R bob:bob "${BOB_LEGACY}"

#
# Show the content of main.py
cat "${BOB_MAIN}"

# Bob's legacy application.
import argparse
import sys

import requests


def create_request(domain):
    """Prepare the request used by the legacy application."""
    return requests.Request(method="GET", url=domain)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--domain", default="https://example.com")
    args = parser.parse_args()

    print("Bob Legacy Application")
    print(f"  Python  : {sys.executable}")
    print(f"  requests: {requests.__version__}")

    request = create_request(args.domain)
    print(f"  request : {request.method} {request.url}")


if __name__ == "__main__":
    main()


With Bob's legacy application in place, its CLI can directly be used using the `python3` system interpreter.

In [23]:
%%bash
set -euo pipefail
python3 /home/bob/projects/legacy-project/main.py --domain "https://example.com"

Bob Legacy Application
  Python  : /usr/bin/python3
  requests: 2.25.1
  request : GET https://example.com


### Alice FastAPI v1

Alice's first application is a small FastAPI web application that exposes a "`/`" endpoint and reports the versions of FastAPI and Requests it is running with. 

Alice's project catches `requests.exceptions.JSONDecodeError` when a service returns invalid JSON. This public exception was added in Requests 2.27, so the project requires a newer Requests release than Bob's application.

Since the notebook runs as Bob, the following `%%bash` cell creates the application in Alice's home directory and assigns ownership to Alice.

In [24]:
%%bash

ALICE_V1="/home/alice/projects/fastapi-v1"
ALICE_MAIN="${ALICE_V1}/main.py"

sudo mkdir -p "${ALICE_V1}"

sudo tee "${ALICE_MAIN}" > /dev/null <<'PYTHON'
# Alice FastAPI project 1
# Uses requests.exceptions.JSONDecodeError, available since Requests 2.27.
import sys

import fastapi
import requests
from fastapi import FastAPI
from requests.exceptions import JSONDecodeError

app = FastAPI(title="alice-fastapi-v1")


def parse_response(response):
    try:
        return response.json()
    # unsupported by "requests==2.25.1"
    except JSONDecodeError:
        return {"error": "The service did not return JSON"}


@app.get("/")
def root():
    response = requests.Response()
    response._content = b"not valid JSON"

    return {
        "project": "alice-fastapi-v1",
        "fastapi": fastapi.__version__,
        "requests": requests.__version__,
        "parsed_response": parse_response(response),
    }


if __name__ == "__main__":
    print("Alice FastAPI v1")
    print(f"  Python  : {sys.executable}")
    print(f"  fastapi : {fastapi.__version__}")
    print(f"  requests: {requests.__version__}")

    response = requests.Response()
    response._content = b"not valid JSON"
    print(f"  result  : {parse_response(response)}")
PYTHON

sudo chown -R alice:alice "${ALICE_V1}"

#
# Show the content of main.py
sudo cat "${ALICE_MAIN}"

# Alice FastAPI project 1
# Uses requests.exceptions.JSONDecodeError, available since Requests 2.27.
import sys

import fastapi
import requests
from fastapi import FastAPI
from requests.exceptions import JSONDecodeError

app = FastAPI(title="alice-fastapi-v1")


def parse_response(response):
    try:
        return response.json()
    # unsupported by "requests==2.25.1"
    except JSONDecodeError:
        return {"error": "The service did not return JSON"}


@app.get("/")
def root():
    response = requests.Response()
    response._content = b"not valid JSON"

    return {
        "project": "alice-fastapi-v1",
        "fastapi": fastapi.__version__,
        "requests": requests.__version__,
        "parsed_response": parse_response(response),
    }


if __name__ == "__main__":
    print("Alice FastAPI v1")
    print(f"  Python  : {sys.executable}")
    print(f"  fastapi : {fastapi.__version__}")
    print(f"  requests: {requests.__version__}")

    response = requests.Response()
    r

### Alice FastAPI v2

Alice's second application is a newer FastAPI project. It uses Pydantic v2's `model_dump()` method and the newer Requests `JSONDecodeError` exception.

Like the first project, the following `%%bash` cell creates it directly in Alice's project directory and assigns ownership to Alice.

In [25]:
%%bash

ALICE_V2="/home/alice/projects/fastapi-v2"
ALICE_MAIN="${ALICE_V2}/main.py"

sudo mkdir -p "${ALICE_V2}"

sudo tee "${ALICE_MAIN}" > /dev/null <<'PYTHON'
# Alice FastAPI project 2
# Uses Pydantic 2 and requests.exceptions.JSONDecodeError.
import sys

import fastapi
import requests
from fastapi import FastAPI
from pydantic import BaseModel
from requests.exceptions import JSONDecodeError

app = FastAPI(title="alice-fastapi-v2")


class Project(BaseModel):
    name: str
    version: str


def parse_response(response):
    try:
        return response.json()
    except JSONDecodeError:
        return {"error": "The service did not return JSON"}


@app.get("/")
def root():
    project = Project(name="alice-fastapi-v2", version=fastapi.__version__)
    response = requests.Response()
    response._content = b"not valid JSON"

    return {
        "project": project.model_dump(),
        "fastapi": fastapi.__version__,
        "requests": requests.__version__,
        "parsed_response": parse_response(response),
    }


if __name__ == "__main__":
    print("Alice FastAPI v2")
    print(f"  Python  : {sys.executable}")
    print(f"  fastapi : {fastapi.__version__}")
    print(f"  requests: {requests.__version__}")

    project = Project(name="alice-fastapi-v2", version=fastapi.__version__)
    print(f"  model   : {project.model_dump()}")

    response = requests.Response()
    response._content = b"not valid JSON"
    print(f"  result  : {parse_response(response)}")
PYTHON

sudo chown -R alice:alice "${ALICE_V2}"

#
# Show the content of main.py
sudo cat $ALICE_MAIN

# Alice FastAPI project 2
# Uses Pydantic 2 and requests.exceptions.JSONDecodeError.
import sys

import fastapi
import requests
from fastapi import FastAPI
from pydantic import BaseModel
from requests.exceptions import JSONDecodeError

app = FastAPI(title="alice-fastapi-v2")


class Project(BaseModel):
    name: str
    version: str


def parse_response(response):
    try:
        return response.json()
    except JSONDecodeError:
        return {"error": "The service did not return JSON"}


@app.get("/")
def root():
    project = Project(name="alice-fastapi-v2", version=fastapi.__version__)
    response = requests.Response()
    response._content = b"not valid JSON"

    return {
        "project": project.model_dump(),
        "fastapi": fastapi.__version__,
        "requests": requests.__version__,
        "parsed_response": parse_response(response),
    }


if __name__ == "__main__":
    print("Alice FastAPI v2")
    print(f"  Python  : {sys.executable}")
    print(f"  fastapi : {f

---

## Unveil Dependency Conflicts

### Unveil the Shared System Package Conflict

Bob and Alice both run the same system interpreter, so they also import packages from the same globally shared `site-packages` directory.

Bob's application works with Requests 2.25.1. Alice's application uses `requests.exceptions.JSONDecodeError`, which was added in Requests 2.27.

In [26]:
%%bash
set -euo pipefail
python3 /home/bob/projects/legacy-project/main.py --domain "https://example.com"

Bob Legacy Application
  Python  : /usr/bin/python3
  requests: 2.25.1
  request : GET https://example.com


We switch to Alice (`sudo -u alice`) before starting the application and using the system interpreter and will run into a Python `ImportError`.

In [27]:
%%bash
sudo -u alice -H python3 /home/alice/projects/fastapi-v1/main.py

Alice FastAPI v1
  Python  : /usr/bin/python3
  fastapi : 0.68.2
  requests: 2.32.3
  result  : {'error': 'The service did not return JSON'}


### Isolating The `requests` Package by User

Remove the shared Requests package now that it has demonstrated the conflict. The first code block cleans up the shared installation and installs Bob's required `requests==2.25.1` into his user site. The second code block verifies the location and runs Bob's application.

In [37]:
%%bash
set -euo pipefail
echo "Remove the shared Requests installation from the system interpreter"
sudo -H python3 -m pip uninstall --break-system-packages --yes requests

echo
echo "Install Bob's required Requests version in his user site"
python3 -m pip install --user "requests==2.25.1"

Remove the shared Requests installation from the system interpreter



Install Bob's required Requests version in his user site


In [39]:
%%bash
set -euo pipefail
echo "Run Bob's legacy application"
python3 /home/bob/projects/legacy-project/main.py

Run Bob's legacy application
Bob Legacy Application
  Python  : /usr/bin/python3
  requests: 2.25.1
  request : GET https://example.com


### Shared Python User Environment Conflicts

Alice installs the newer Requests release in her own user site. The first code block installs `requests==2.32.3` and verifies where it is loaded from. The second code block runs Alice's application with that user-level package set. Bob's package remains untouched, but both of Alice's projects still share this single user-level environment.

In [42]:
%%bash
set -euo pipefail

echo "Install Alice's required Requests version in her user site"
sudo -u alice -H python3 -m pip install --user --upgrade "requests==2.32.3"

Install Alice's required Requests version in her user site


In [43]:
%%bash
set -euo pipefail
echo "Run Alice's FastAPI v1 application"
sudo -u alice -H python3 /home/alice/projects/fastapi-v1/main.py

Run Alice's FastAPI v1 application
Alice FastAPI v1
  Python  : /usr/bin/python3
  fastapi : 0.111.1
  requests: 2.32.3
  result  : {'error': 'The service did not return JSON'}


> The shared Requests copy is gone. Bob now imports 2.25.1 from his user site, while Alice imports 2.32.3 from hers. `pip install --user` isolates the users, but Alice's two projects still share the same `~/.local` package directory.

### Unveil the Shared User Package Conflict

Start with Alice's older FastAPI dependency set. The first code block installs the older shared user dependencies. The second code block runs `alice-fastapi-v1`. The following code block then runs `alice-fastapi-v2`, which fails because that same shared user environment still contains the older FastAPI and Pydantic versions.

In [48]:
%%bash
set -euo pipefail
echo "Install the older FastAPI and Pydantic versions in Alice's shared user site"
sudo -u alice -H python3 -m pip install --user --upgrade "fastapi==0.68.2" "pydantic<2"

Install the older FastAPI and Pydantic versions in Alice's shared user site


In [49]:
%%bash
set -euo pipefail
echo "Run alice-fastapi-v1 with the shared user environment"
sudo -u alice -H python3 /home/alice/projects/fastapi-v1/main.py

Run alice-fastapi-v1 with the shared user environment
Alice FastAPI v1
  Python  : /usr/bin/python3
  fastapi : 0.68.2
  requests: 2.32.3
  result  : {'error': 'The service did not return JSON'}


In [50]:
%%bash
set -euo pipefail
echo "Run alice-fastapi-v2 with the same shared user environment"
sudo -u alice -H python3 /home/alice/projects/fastapi-v2/main.py || true

Run alice-fastapi-v2 with the same shared user environment
Alice FastAPI v2
  Python  : /usr/bin/python3
  fastapi : 0.68.2
  requests: 2.32.3


Traceback (most recent call last):
  File "/home/alice/projects/fastapi-v2/main.py", line 47, in <module>
    print(f"  model   : {project.model_dump()}")
AttributeError: 'Project' object has no attribute 'model_dump'


### Reparing the Shared User Package Conflict

The `--user` flag separates packages by user, but Alice's projects still share the same `~/.local` directory. That means two projects with incompatible FastAPI dependencies can still break each other.

The fix is to give each Alice project its own virtual environment. Bob can keep using his user-level Requests installation.

In [52]:
%%bash
set -euo pipefail
echo "Create the virtual environment for alice-fastapi-v1"
sudo -u alice -H python3 -m venv /home/alice/projects/fastapi-v1/.venv
echo "Upgrade pip inside alice-fastapi-v1/.venv"
sudo -u alice -H /home/alice/projects/fastapi-v1/.venv/bin/python -m pip install --quiet --upgrade pip
echo "Install project dependencies for alice-fastapi-v1"
sudo -u alice -H /home/alice/projects/fastapi-v1/.venv/bin/python -m pip install --quiet "fastapi==0.68.2" "pydantic<2" "requests==2.32.3"

Create the virtual environment for alice-fastapi-v1
Upgrade pip inside alice-fastapi-v1/.venv
Install project dependencies for alice-fastapi-v1


In [53]:
%%bash
set -euo pipefail
echo "Create the virtual environment for alice-fastapi-v2"
sudo -u alice -H python3 -m venv /home/alice/projects/fastapi-v2/.venv
echo "Upgrade pip inside alice-fastapi-v2/.venv"
sudo -u alice -H /home/alice/projects/fastapi-v2/.venv/bin/python -m pip install --quiet --upgrade pip
echo "Install project dependencies for alice-fastapi-v2"
sudo -u alice -H /home/alice/projects/fastapi-v2/.venv/bin/python -m pip install --quiet "fastapi==0.111.1" "requests==2.32.3"

Create the virtual environment for alice-fastapi-v2
Upgrade pip inside alice-fastapi-v2/.venv
Install project dependencies for alice-fastapi-v2


### The final state: three projects, isolated dependencies

Bob now runs with his user-level Requests installation, while Alice's two FastAPI projects run with their own `.venv` interpreters. The output below should show independent executable paths and dependency versions.

In [54]:
%%bash
set -euo pipefail
echo 'Bob Legacy'
echo '----------'
python3 /home/bob/projects/legacy-project/main.py
echo
echo 'Alice FastAPI V1'
echo '----------------'
sudo su - alice -c '/home/alice/projects/fastapi-v1/.venv/bin/python /home/alice/projects/fastapi-v1/main.py'
echo
echo 'Alice FastAPI V2'
echo '----------------'
sudo su - alice -c '/home/alice/projects/fastapi-v2/.venv/bin/python /home/alice/projects/fastapi-v2/main.py'

Bob Legacy
----------
Bob Legacy Application
  Python  : /usr/bin/python3
  requests: 2.25.1
  request : GET https://example.com

Alice FastAPI V1
----------------
Alice FastAPI v1
  Python  : /home/alice/projects/fastapi-v1/.venv/bin/python
  fastapi : 0.68.2
  requests: 2.32.3
  result  : {'error': 'The service did not return JSON'}

Alice FastAPI V2
----------------
Alice FastAPI v2
  Python  : /home/alice/projects/fastapi-v2/.venv/bin/python
  fastapi : 0.111.1
  requests: 2.32.3
  model   : {'name': 'alice-fastapi-v2', 'version': '0.111.1'}
  result  : {'error': 'The service did not return JSON'}
